In [2]:
import pandas as pd
df = pd.read_csv("amazon_audio.csv")

In [2]:
df.dtypes

asin                 object
product_title        object
brand                object
price               float64
discount_percent     object
rating              float64
review_count         object
category             object
product_link         object
search_term          object
dtype: object

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   asin              1500 non-null   object 
 1   product_title     1500 non-null   object 
 2   brand             1500 non-null   object 
 3   price             1453 non-null   float64
 4   discount_percent  1391 non-null   object 
 5   rating            1369 non-null   float64
 6   review_count      1369 non-null   object 
 7   category          1500 non-null   object 
 8   product_link      1500 non-null   object 
 9   search_term       1500 non-null   object 
dtypes: float64(2), object(8)
memory usage: 117.3+ KB


In [4]:
df.describe()

,price,rating
count,1453.000000,1369.000000
mean,7413.721067,3.926150
std,27012.539563,0.657416
min,89.000000,1.000000
25%,599.000000,3.700000
50%,1299.000000,4.000000
75%,4299.000000,4.300000
max,442425.000000,5.000000


### Handling duplicate values

In [3]:
df.duplicated().sum()

np.int64(0)

###  Data types convertion

In [4]:
# For discount
df['discount_percent'] = df['discount_percent'].str.extract(r'(\d+)').astype(float)

#### Vectorization

In [5]:
# Checking what kind of var. are there.
df['review_count'].unique()

array(['21.6K', '30.6K', '4.4K', '256', nan, '27.8K', '303', '28K',
       '1.4K', '14.9K', '92', '9.9K', '4.3K', '66.9K', '82.4K', '10.5K',
       '343', '1.1K', '1.3K', '3.8K', '20.2K', '3.5K', '5.5K', '3.1K',
       '3.5L', '99.9K', '423', '11.6K', '2L', '678', '16.1K', '4.9K',
       '223', '13', '16.3K', '32.9K', '1.2K', '240', '2.3K', '569', '122',
       '4.1K', '345', '3.2K', '74', '2.2L', '36.5K', '4.3L', '2.2K',
       '68.2K', '173', '50', '1.3L', '1.5K', '4.7K', '26.8K', '422', '65',
       '66K', '58', '27.7K', '24.1K', '5', '54', '8.2K', '7K', '7.9K',
       '10.4K', '1.7K', '12.2K', '260', '11.2K', '156', '607', '1.1L',
       '707', '40.6K', '112', '6.2K', '323', '5.3K', '17', '23', '672',
       '479', '5.1K', '819', '440', '26', '8.6K', '587', '15.4K', '18.4K',
       '16.2K', '403', '654', '8.9K', '9', '509', '90', '907', '283',
       '649', '162', '176', '216', '274', '63.5K', '15.3K', '2.4K', '123',
       '48', '37.4K', '43', '508', '316', '9.3K', '584', '67.4K',

In [6]:
import pandas as pd

df['review_count'] = (
    df['review_count']
    .str.lower()
    .str.replace('k', 'e3')
    .str.replace('l', 'e5')
    .astype(float)
)

### Dropping 

In [7]:
# Dropping rows which has only numbers
df = df[~df['brand'].str.match(r'^\d+$', na=False)]

In [8]:
# Dropping rows with missing values in the 'Price' column,
# Imputing with mean or median is not appropriate for price data.
df['price'].isnull().sum()

np.int64(47)

In [9]:
# Dropping
df = df.dropna(subset=['price'])

### Cleaning object and numeric columns

In [10]:
# Assing object to obj_cols
obj_cols = df.select_dtypes(include = 'object').columns

In [11]:
# Assing object to num_col 
num_cols = df.select_dtypes(include = 'float').columns

#### Cleaning both object and numeric columns

In [12]:
# Treating object columns
# lowering, eliminating unwanted spaces, removing special characters
for col in obj_cols:
    if col not in ['asin', 'product_link']:
        df[col] = df[col].str.lower().str.strip().str.replace(r'[^a-z0-9\s]', "", regex = True)

In [13]:
# Treating numeric columns
# For rating - median(), other with 0
for num in num_cols:
    if num == 'price':
        continue
    elif num == 'rating':
        df[num] = df[num].fillna(df[num].median())
    else:
        df[num] = df[num].fillna(0)

### Feature extraction

In [14]:
# The value of the product on price
# High rating + Low price → High value_score → High rank
# Low rating + High price → Low value_score → Low rank
df['value_score'] = df['rating'] / df['price']

In [15]:
# Checking which kind of device it is
device_keywords = {
    "earbuds": ["earbud", "earbuds", "tws", "buds", "pods"],
    "neckband": ["neckband"],
    "headphones": ["headphone", "headphones", "over ear", "on ear", "headset"],
    "earphones": ["earphone", "earphones", "in ear", "wired"],
    "speakers": ["speaker", "speakers", "bluetooth speaker", "wireless speaker", "home speaker", "portable speaker", "soundbar", "smart speaker"]
}

priority_order = ["earbuds", "neckband", "headphones", "earphones", "speakers"]

def detect_device_type(title):
    title = title.lower()

    for device in priority_order:
        for keyword in device_keywords[device]:
            if keyword in title:
                return device

    return "unknown"

In [16]:
# Applying
df['device_type'] = df['product_title'].apply(detect_device_type)

In [17]:
df['device_type'].value_counts()

device_type
earbuds       568
headphones    314
speakers      289
neckband      173
earphones      99
unknown         5
Name: count, dtype: int64

In [18]:
# Wired -  1
# Wireless - 0
df['is_wired'] = df['product_title'].str.lower().str.contains(
    r'\b(wired|aux|3\.5mm|jack)\b',
    na=False
).astype(int)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_18192\1234706907.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['is_wired'] = df['product_title'].str.lower().str.contains(


In [19]:
df['is_wired'].value_counts()

is_wired
0    1137
1     311
Name: count, dtype: int64

In [20]:
# has mic - 1
# no mic - 0
df['has_mic'] = df['product_title'].str.lower().str.contains(
    r'\b(mic|microphone|hands[- ]?free|calling)\b',
    na=False
).astype(int)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_18192\1959774548.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['has_mic'] = df['product_title'].str.lower().str.contains(


In [22]:
df['has_mic'].value_counts()

has_mic
0    769
1    679
Name: count, dtype: int64

### Checking

In [21]:
df.columns

Index(['asin', 'product_title', 'brand', 'price', 'discount_percent', 'rating',
       'review_count', 'category', 'product_link', 'search_term',
       'value_score', 'device_type', 'is_wired', 'has_mic'],
      dtype='object')

In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1448 entries, 0 to 1499
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   asin              1448 non-null   object 
 1   product_title     1448 non-null   object 
 2   brand             1448 non-null   object 
 3   price             1448 non-null   float64
 4   discount_percent  1448 non-null   float64
 5   rating            1448 non-null   float64
 6   review_count      1448 non-null   float64
 7   category          1448 non-null   object 
 8   product_link      1448 non-null   object 
 9   search_term       1448 non-null   object 
 10  value_score       1448 non-null   float64
 11  device_type       1448 non-null   object 
 12  is_wired          1448 non-null   int64  
 13  has_mic           1448 non-null   int64  
dtypes: float64(5), int64(2), object(7)
memory usage: 169.7+ KB


In [23]:
df.describe()

,price,discount_percent,rating,review_count,value_score,is_wired,has_mic
count,1448.000000,1448.000000,1448.000000,1448.000000,1448.000000,1448.000000,1448.000000
mean,7248.648971,52.561464,3.934254,6107.694751,0.004579,0.214779,0.468923
std,26260.344133,22.777590,0.623321,26331.310520,0.005058,0.410811,0.499206
min,89.000000,0.000000,1.000000,0.000000,0.000009,0.000000,0.000000
25%,599.000000,38.000000,3.800000,13.000000,0.000938,0.000000,0.000000
50%,1299.000000,57.000000,4.000000,251.500000,0.002925,0.000000,0.000000
75%,4299.000000,70.000000,4.200000,1900.000000,0.006180,0.000000,1.000000
max,442425.000000,94.000000,5.000000,430000.000000,0.046067,1.000000,1.000000


In [24]:
df.isnull().sum()

asin                0
product_title       0
brand               0
price               0
discount_percent    0
rating              0
review_count        0
category            0
product_link        0
search_term         0
value_score         0
device_type         0
is_wired            0
has_mic             0
dtype: int64

### Saving to csv

In [23]:
df.to_csv("cleaned_amazon_audio.csv", index=False)